# 08a — HOME coverage expansion / confidence tiers

Stage 07p closed the validated production core:

~~~text
HOME_HIGH_CONFIDENCE_CORE = 27
OFFICE_HIGH_CONFIDENCE_CORE = 16
~~~

Stage 08a starts a new coverage-expansion track. It does not reopen or weaken the validated HOME gate.

Stage 05b already found only two non-production HOME candidates that were unique consensus winners at HIGH/MEDIUM internal evidence tier.

Therefore 08a is a targeted audit of those two cases, augmented with independent evidence measured later:

~~~text
Stage 05b internal consensus + reliability
        ↓
2 non-production HIGH/MEDIUM HOME candidates
        ↓
Trackintel OSNA/FREQ on frozen namespace (07m)
scikit-mobility-style HOME (07o)
SCITEPRESS-style HOME (07o)
geohash HOME spatial support (07o)
Trackintel end-to-end HOME spatial support (07n)
        ↓
HOME_PROBABLE / HOME_PLAUSIBLE / ABSTAIN
~~~

These are evidence tiers, not calibrated probabilities or semantic ground truth.

The production HOME count remains 27 in this stage.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
V=Path('/mnt/geolife-data')
ROOT=V/'cache'/'cp2_v2'
C=ROOT/'08a_home_coverage_expansion'
C.mkdir(parents=True,exist_ok=True)

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)

for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

spec=importlib.util.spec_from_file_location(
    's08a',
    REPO_DIR/'analysis'/'08a_home_coverage_expansion.py',
)
assert spec is not None and spec.loader is not None
s08a=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s08a
spec.loader.exec_module(s08a)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print(s08a.synthetic_self_check())

## 1. Frozen production + Stage-05b gates

The production core must still reproduce HOME 27 / OFFICE 16.

Stage 05b must still yield exactly two non-production HIGH/MEDIUM unique HOME consensus winners.

In [ ]:
def find_stay_cache():
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
        REPO_DIR/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for p in candidates:
        if p.exists():
            return p
    hits=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if hits:
        return hits[0]
    raise FileNotFoundError('stays_baseline_v1.pkl')

stays=pd.read_pickle(find_stay_cache())
cfg=HomeOfficeConfig()
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
production=infer_home_office(stays,config=cfg)

counts=production['label'].value_counts().to_dict()
assert len(stays)==5821
assert counts.get('HOME',0)==27
assert counts.get('OFFICE',0)==16

HOME_05B=ROOT/'05b_home_consensus_adaptive_work'/'home_consensus_private.pkl'
if not HOME_05B.exists():
    raise FileNotFoundError(HOME_05B)

home_evidence=pd.read_pickle(HOME_05B)
expansion_candidates=s08a.select_expansion_candidates(home_evidence)
assert len(expansion_candidates)==2

expansion_candidates=s08a.attach_location_context(
    expansion_candidates,
    locations,
    production,
)

gate=pd.DataFrame([{
    'cp1_stays':len(stays),
    'production_HOME':counts.get('HOME',0),
    'production_OFFICE':counts.get('OFFICE',0),
    'stage05b_high_medium_expansion_candidates':len(expansion_candidates),
}])
print('production + Stage-05b expansion gate: PASS')
display(gate)

## 2. Load independent later-stage evidence

Required private caches:

~~~text
05b_home_consensus_adaptive_work/home_consensus_private.pkl
07m_trackintel_semantic_parity/trackintel_candidates_private.pkl
07n_trackintel_end_to_end/trackintel_osna_candidates_private.pkl
07o_literature_comparator_suite/scikit_home_candidates_private.pkl
07o_literature_comparator_suite/scitepress_candidates_private.pkl
07o_literature_comparator_suite/geohash_user_home_private.pkl
~~~

The same-namespace exact confirmations are intentionally restricted to three different semantic families for HOME_PROBABLE:

1. Trackintel OSNA;
2. scikit nighttime-frequency HOME;
3. SCITEPRESS work/rest-style HOME.

Trackintel FREQ, geohash, and end-to-end Trackintel remain auxiliary evidence.

In [ ]:
paths={
    'trackintel_semantic':ROOT/'07m_trackintel_semantic_parity'/'trackintel_candidates_private.pkl',
    'trackintel_e2e':ROOT/'07n_trackintel_end_to_end'/'trackintel_osna_candidates_private.pkl',
    'scikit':ROOT/'07o_literature_comparator_suite'/'scikit_home_candidates_private.pkl',
    'scitepress':ROOT/'07o_literature_comparator_suite'/'scitepress_candidates_private.pkl',
    'geohash':ROOT/'07o_literature_comparator_suite'/'geohash_user_home_private.pkl',
}
missing=[str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Stage-08a prerequisite caches:\n'+'\n'.join(missing))

trackintel_semantic=pd.read_pickle(paths['trackintel_semantic'])
trackintel_e2e=pd.read_pickle(paths['trackintel_e2e'])
scikit_home=pd.read_pickle(paths['scikit'])
scitepress=pd.read_pickle(paths['scitepress'])
geohash_home=pd.read_pickle(paths['geohash'])

cache_inventory=pd.DataFrame([
    {'source':name,'rows':len(frame)}
    for name,frame in [
        ('07m Trackintel semantic',trackintel_semantic),
        ('07n Trackintel end-to-end',trackintel_e2e),
        ('07o scikit HOME',scikit_home),
        ('07o SCITEPRESS',scitepress),
        ('07o geohash HOME',geohash_home),
    ]
])
display(cache_inventory)

## 3. Build two-candidate evidence matrix

No total weighted score is created.

The matrix keeps internal reliability, exact independent confirmations, spatial support, and OFFICE collision as separate columns.

In [ ]:
matrix=s08a.build_expansion_evidence_matrix(
    expansion_candidates,
    trackintel_semantic_candidates=trackintel_semantic,
    scikit_home_candidates=scikit_home,
    scitepress_candidates=scitepress,
    geohash_home_candidates=geohash_home,
    trackintel_e2e_candidates=trackintel_e2e,
)
tiered=s08a.assign_expansion_tiers(matrix)

tiered=tiered.sort_values('user_id',kind='stable').reset_index(drop=True)
tiered.insert(0,'audit_id',[f'H{i:02d}' for i in range(1,len(tiered)+1)])

private_map=tiered[['audit_id','user_id','location_id']].copy()
review_cols=[
    'audit_id',
    'home_tier',
    'method_votes',
    'reliability_axes',
    'split_axes_confirmed',
    'holdout_top1_method_count',
    'dropout30_mean_retention',
    'trackintel_osna_exact',
    'trackintel_freq_exact',
    'scikit_home_exact',
    'scitepress_home_exact',
    'external_exact_family_count',
    'geohash_within_200m',
    'geohash_distance_m',
    'trackintel_e2e_dbscan100_within_200m',
    'trackintel_e2e_dbscan100_distance_m',
    'trackintel_e2e_dbscan200_within_200m',
    'trackintel_e2e_dbscan200_distance_m',
    'collides_with_production_office',
    'home_expansion_tier',
    'tier_reason',
]
review=tiered[[c for c in review_cols if c in tiered.columns]].copy()

print('TARGETED HOME EXPANSION EVIDENCE')
display(review)

## 4. Predeclared tier contract

### HOME_PROBABLE

A candidate must satisfy all of:

~~~text
Stage-05b tier in {HIGH, MEDIUM}
unique vote winner
method_votes >= 2
reliability_axes >= 2

AND

at least 2 exact confirmations among:
    Trackintel OSNA
    scikit HOME
    SCITEPRESS HOME

AND

candidate != production OFFICE location
~~~

### HOME_PLAUSIBLE

Same internal evidence floor, no OFFICE collision, but not enough for PROBABLE.

It still needs at least one of:

- one exact external-family confirmation;
- Trackintel FREQ exact;
- geohash within 200 m;
- Trackintel end-to-end HOME within 200 m.

### ABSTAIN

Anything weaker or semantically conflicting.

These tiers do not change the production gate.

In [ ]:
support_summary=s08a.candidate_support_summary(tiered)
coverage_summary=s08a.summarize_expansion(tiered,core_home_users=27)

print('HOME EXPANSION SUPPORT SUMMARY')
display(support_summary)

print('HOME TIERED COVERAGE')
display(coverage_summary)

## 5. Candidate descriptive location context

Pavan-style intensity/frequency was useful as descriptive context in Stage 07o, but it is not part of the promotion rule.

Show the two candidate locations' production-location support without turning these features into a score.

In [ ]:
descriptive_cols=[
    'audit_id',
    'home_tier',
    'stay_count',
    'total_dwell_h',
    'active_local_dates',
    'diameter_m',
    'home_dwell_h',
    'home_share',
    'home_expansion_tier',
]
descriptive=tiered[[c for c in descriptive_cols if c in tiered.columns]].copy()
display(descriptive)

## 6. Policy boundary

Stage 08a creates a tiered coverage layer on top of the frozen core.

~~~text
HOME_HIGH_CONFIDENCE_CORE = 27
HOME_PROBABLE_NEW         = measured from the 2 targeted candidates
HOME_PLAUSIBLE_NEW        = measured from the 2 targeted candidates
ABSTAIN                    = targeted candidates lacking sufficient support
~~~

Production remains:

~~~text
HOME = 27
OFFICE = 16
~~~

A HOME_PROBABLE result may be exposed downstream as a separate confidence tier, but it must not be silently merged into production HOME.

In [ ]:
policy=pd.DataFrame([{
    'HOME_HIGH_CONFIDENCE_CORE':27,
    'HOME_PROBABLE_NEW':int(tiered['home_expansion_tier'].eq('HOME_PROBABLE').sum()),
    'HOME_PLAUSIBLE_NEW':int(tiered['home_expansion_tier'].eq('HOME_PLAUSIBLE').sum()),
    'HOME_EXPANSION_ABSTAIN':int(tiered['home_expansion_tier'].eq('ABSTAIN').sum()),
    'production_HOME':27,
    'production_OFFICE':16,
    'production_home_gate_changed':False,
    'production_office_gate_changed':False,
    'tiered_expansion_enabled':True,
}])
display(policy)

## 7. Save Stage-08a outputs

Candidate identities remain private on the Modal Volume.

Aggregate outputs report only counts and support summaries.

In [ ]:
private_map.to_pickle(C/'home_expansion_audit_map_private.pkl')
tiered.to_pickle(C/'home_expansion_evidence_private.pkl')

gate.to_csv(C/'frozen_and_candidate_gate.csv',index=False)
cache_inventory.to_csv(C/'external_cache_inventory.csv',index=False)
support_summary.to_csv(C/'home_expansion_support_summary.csv',index=False)
coverage_summary.to_csv(C/'home_tiered_coverage_summary.csv',index=False)
policy.to_csv(C/'home_expansion_policy.csv',index=False)

print('saved:',C)

## Interpretation boundary

- Stage 08a audits exactly the two HIGH/MEDIUM non-production HOME candidates already identified by Stage 05b.
- It does not search all 109 non-HOME users indiscriminately.
- HOME_PROBABLE and HOME_PLAUSIBLE are transparent evidence tiers, not calibrated probabilities.
- Trackintel/scikit/SCITEPRESS agreement is triangulation, not ground truth.
- Geohash and Trackintel end-to-end spatial support are auxiliary because they use different spatial representations.
- Production HOME remains 27 until a separate policy decision explicitly changes the production contract.
- The intended use of a positive 08a result is a tiered downstream label, not silent threshold relaxation.